In [4]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/muna/Hana_research/data/db/orca_receipt.db"
CHAR_FFFD = "\ufffd"

with sqlite3.connect(DB_PATH) as conn:

    # 置換文字を含む行の件数
    q1 = """
    SELECT COUNT(*) AS n_rows
    FROM orca_receipt_details
    WHERE detail_name          LIKE '%' || char(65533) || '%'
       OR official_detail_name LIKE '%' || char(65533) || '%'
       OR unit_name            LIKE '%' || char(65533) || '%'
    """
    print(pd.read_sql_query(q1, conn))

    # 該当する薬剤・名称の内訳
    q2 = """
    SELECT detail_code, drug_price_code, detail_name, COUNT(*) AS n
    FROM orca_receipt_details
    WHERE detail_name LIKE '%' || char(65533) || '%'
    GROUP BY detail_code, drug_price_code, detail_name
    ORDER BY n DESC
    """
    df = pd.read_sql_query(q2, conn)
    print(df.head(50))

   n_rows
0       0
Empty DataFrame
Columns: [detail_code, drug_price_code, detail_name, n]
Index: []


取り込み済みの DB の修正　〜のところ

In [3]:
import sqlite3

DB_PATH = "/Users/muna/Hana_research/data/db/orca_receipt.db"

with sqlite3.connect(DB_PATH) as conn:
    for col in ["detail_name", "official_detail_name", "unit_name"]:
        conn.execute(
            f"UPDATE orca_receipt_details "
            f"SET {col} = REPLACE({col}, char(65533) || 'ー', '〜') "
            f"WHERE {col} LIKE '%' || char(65533) || '%'"
        )
    conn.commit()

    left = conn.execute(
        "SELECT COUNT(*) FROM orca_receipt_details "
        "WHERE detail_name LIKE '%' || char(65533) || '%' "
        "OR official_detail_name LIKE '%' || char(65533) || '%' "
        "OR unit_name LIKE '%' || char(65533) || '%'"
    ).fetchone()[0]
    print("残存件数:", left)

残存件数: 0


patient_no→ Patient_IDに修正．

In [5]:
import sqlite3

DB_PATH = "/Users/muna/Hana_research/data/db/orca_receipt.db"

with sqlite3.connect(DB_PATH) as conn:
    print("SQLite:", sqlite3.sqlite_version)
    for row in conn.execute("PRAGMA table_info(orca_receipt_details)"):
        print(row[1])

SQLite: 3.52.0
id
billing_month
service_date
patient_no
drug_group_no
detail_seq
service_category
service_type
detail_code
detail_quantity
detail_count
unit_code
unit_name
detail_name
official_detail_name
drug_price_code
daily_value
source_file
source_row_number


In [6]:
import shutil
from datetime import datetime

backup = DB_PATH.replace(".db", f"_backup_{datetime.now():%Y%m%d_%H%M%S}.db")
shutil.copy2(DB_PATH, backup)
print("バックアップ:", backup)

バックアップ: /Users/muna/Hana_research/data/db/orca_receipt_backup_20261013_145642.db


In [7]:
with sqlite3.connect(DB_PATH) as conn:
    cols = [r[1] for r in conn.execute("PRAGMA table_info(orca_receipt_details)")]

    if "Patient_ID" in cols:
        print("すでに Patient_ID です。変更不要。")
    elif "patient_no" in cols:
        conn.execute(
            "ALTER TABLE orca_receipt_details "
            "RENAME COLUMN patient_no TO Patient_ID"
        )
        conn.commit()
        print("patient_no → Patient_ID に変更しました。")
    else:
        raise RuntimeError(f"想定外の列構成です: {cols}")

    # 確認
    print([r[1] for r in conn.execute("PRAGMA table_info(orca_receipt_details)")])

patient_no → Patient_ID に変更しました。
['id', 'billing_month', 'service_date', 'Patient_ID', 'drug_group_no', 'detail_seq', 'service_category', 'service_type', 'detail_code', 'detail_quantity', 'detail_count', 'unit_code', 'unit_name', 'detail_name', 'official_detail_name', 'drug_price_code', 'daily_value', 'source_file', 'source_row_number']
